In [ ]:
library(DBI)
library(dplyr)
library(tidyr)

con <- dbConnect(
  drv = RMariaDB::MariaDB(),
  username = "YOUR_USERNAME",
  password = "YOUR_PASSWORD",
  host = "YOUR_HOST_ADDRESS",
  dbname = "mimiciiiv14",
  port = 3306
)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




The query retrieves adult patients’ first ICU stay of at least 24 hours, including patient ID, hospital admission ID, ICU stay ID, ICU admission time, gender, and age.

In [2]:
N_PATIENTS <- 32000


# Build SQL query to define the ICU cohort
cohort_sql <- paste0("
WITH first_icu AS (
    SELECT
        icu.subject_id,
        icu.hadm_id,
        icu.icustay_id,
        icu.intime,
        icu.outtime,
        ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) AS rn
    FROM ICUSTAYS icu
),
cohort AS (
    SELECT
        f.subject_id,
        f.hadm_id,
        f.icustay_id,
        f.intime,
        p.gender,
        YEAR(f.intime) - YEAR(p.dob) AS age,
        a.hospital_expire_flag   -- <--- Aquí agregamos la variable
    FROM first_icu f
    JOIN PATIENTS p
        ON f.subject_id = p.subject_id
    JOIN ADMISSIONS a
        ON f.hadm_id = a.hadm_id
    WHERE
        f.rn = 1
        AND TIMESTAMPDIFF(HOUR, f.intime, f.outtime) >= 24
        AND (YEAR(f.intime) - YEAR(p.dob)) >= 18
)
SELECT *
FROM cohort
LIMIT ", N_PATIENTS, ";
")

# Display the first rows
cohort_base <- dbGetQuery(con, cohort_sql)
head(cohort_base)



,subject_id,hadm_id,icustay_id,intime,gender,age,hospital_expire_flag
,<int>,<int>,<int>,<dttm>,<chr>,<int>,<int>
1,3,145834,211552,2101-10-20 19:10:11,M,76,0
2,4,185777,294638,2191-03-16 00:29:31,F,48,0
3,6,107064,228232,2175-05-30 21:30:54,F,66,0
4,9,150750,220597,2149-11-09 13:07:02,M,41,1
5,11,194540,229441,2178-04-16 06:19:32,F,50,0
6,12,112213,232669,2104-08-08 02:08:17,M,72,1


This query extracts summary statistics (mean, minimum, and maximum) of key vital signs during the first 24 hours of each ICU stay, processing ICU stays in blocks to manage query size and performance.

In [3]:
icu_ids <- cohort_base$icustay_id
total <- length(icu_ids)
cat("Total ICU stays to process for vitals:", total, "\n")
flush.console()

results <- list()
block_size <- 200

for (i in seq(1, total, by = block_size)) {
  block <- icu_ids[i:min(i + block_size - 1, total)]
  
  sql_block <- paste0("
    SELECT
      c.icustay_id,
      AVG(CASE WHEN ce.itemid IN (211,220045) THEN ce.valuenum END) AS hr_mean,
      MIN(CASE WHEN ce.itemid IN (211,220045) THEN ce.valuenum END) AS hr_min,
      MAX(CASE WHEN ce.itemid IN (211,220045) THEN ce.valuenum END) AS hr_max,
      
      AVG(CASE WHEN ce.itemid IN (456,52,6702,220052) THEN ce.valuenum END) AS map_mean,
      MIN(CASE WHEN ce.itemid IN (456,52,6702,220052) THEN ce.valuenum END) AS map_min,
      MAX(CASE WHEN ce.itemid IN (456,52,6702,220052) THEN ce.valuenum END) AS map_max,
      
      AVG(CASE WHEN ce.itemid IN (618,220210) THEN ce.valuenum END) AS rr_mean,
      MIN(CASE WHEN ce.itemid IN (618,220210) THEN ce.valuenum END) AS rr_min,
      MAX(CASE WHEN ce.itemid IN (618,220210) THEN ce.valuenum END) AS rr_max,
      
      AVG(CASE WHEN ce.itemid IN (646,220277) THEN ce.valuenum END) AS spo2_mean,
      MIN(CASE WHEN ce.itemid IN (646,220277) THEN ce.valuenum END) AS spo2_min,
      MAX(CASE WHEN ce.itemid IN (646,220277) THEN ce.valuenum END) AS spo2_max,
      
      AVG(CASE WHEN ce.itemid IN (223761,678) THEN ce.valuenum END) AS temp_mean,
      MIN(CASE WHEN ce.itemid IN (223761,678) THEN ce.valuenum END) AS temp_min,
      MAX(CASE WHEN ce.itemid IN (223761,678) THEN ce.valuenum END) AS temp_max
    FROM ICUSTAYS c
    JOIN CHARTEVENTS ce
      ON c.icustay_id = ce.icustay_id
    WHERE c.icustay_id IN (", paste(block, collapse = ","), ")
      AND ce.charttime BETWEEN c.intime AND DATE_ADD(c.intime, INTERVAL 24 HOUR)
      AND ce.valuenum IS NOT NULL
    GROUP BY c.icustay_id;
  ")
  
  results[[i]] <- dbGetQuery(con, sql_block)
  
  current <- min(i + block_size - 1, total)
  if (current %% 1000 == 0 || current == total) {
    cat("Processed", current, "of", total, "ICU stays for vitals\n")
    flush.console()
  }
}

vitals_data <- bind_rows(results)
head(vitals_data)


Total ICU stays to process for vitals: 32000 
Processed 1000 of 32000 ICU stays for vitals
Processed 2000 of 32000 ICU stays for vitals
Processed 3000 of 32000 ICU stays for vitals
Processed 4000 of 32000 ICU stays for vitals
Processed 5000 of 32000 ICU stays for vitals
Processed 6000 of 32000 ICU stays for vitals
Processed 7000 of 32000 ICU stays for vitals
Processed 8000 of 32000 ICU stays for vitals
Processed 9000 of 32000 ICU stays for vitals
Processed 10000 of 32000 ICU stays for vitals
Processed 11000 of 32000 ICU stays for vitals
Processed 12000 of 32000 ICU stays for vitals
Processed 13000 of 32000 ICU stays for vitals
Processed 14000 of 32000 ICU stays for vitals
Processed 15000 of 32000 ICU stays for vitals
Processed 16000 of 32000 ICU stays for vitals
Processed 17000 of 32000 ICU stays for vitals
Processed 18000 of 32000 ICU stays for vitals
Processed 19000 of 32000 ICU stays for vitals
Processed 20000 of 32000 ICU stays for vitals
Processed 21000 of 32000 ICU stays for vita

,icustay_id,hr_mean,hr_min,hr_max,map_mean,map_min,map_max,rr_mean,rr_min,rr_max,spo2_mean,spo2_min,spo2_max,temp_mean,temp_min,temp_max
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,200066,100.75000,91,115,81.84848,68.0000,105.0000,19.09091,13,28,97.70000,95,100,97.30000,96.7,97.6
2,201239,92.87500,79,104,71.75000,57.3333,96.6667,18.00000,12,28,97.87500,91,100,98.63333,98.2,99.6
3,202248,86.96667,78,98,85.46667,65.0000,126.0000,13.60000,10,26,99.06250,94,100,99.36667,97.6,101.3
4,203487,72.90625,49,104,63.32433,24.0000,92.0000,16.03125,10,21,97.09677,91,100,97.40000,96.3,98.2
5,204407,83.35000,62,96,54.53334,40.6667,73.6667,15.20000,10,21,96.26829,95,99,96.49091,95.8,98.1
6,204798,101.87500,85,122,96.71793,86.6667,115.0000,20.00000,18,22,98.12500,96,100,98.50000,97.6,99.4


This query computes mean, minimum, and maximum values of selected laboratory measurements (lactate, creatinine, and white blood cell count) during the first 24 hours of each ICU stay.

In [4]:
icu_ids <- cohort_base$icustay_id
total <- length(icu_ids)
cat("Total ICU stays to process for labs:", total, "\n")
flush.console()

results <- list()
block_size <- 200

for (i in seq(1, total, by = block_size)) {
  block <- icu_ids[i:min(i + block_size - 1, total)]
  
  sql_block <- paste0("
    SELECT
      c.icustay_id,
      AVG(CASE WHEN le.itemid = 50813 THEN le.valuenum END) AS lactate_mean,
      MIN(CASE WHEN le.itemid = 50813 THEN le.valuenum END) AS lactate_min,
      MAX(CASE WHEN le.itemid = 50813 THEN le.valuenum END) AS lactate_max,
      
      AVG(CASE WHEN le.itemid = 50912 THEN le.valuenum END) AS creatinine_mean,
      MIN(CASE WHEN le.itemid = 50912 THEN le.valuenum END) AS creatinine_min,
      MAX(CASE WHEN le.itemid = 50912 THEN le.valuenum END) AS creatinine_max,
      
      AVG(CASE WHEN le.itemid = 51300 THEN le.valuenum END) AS wbc_mean,
      MIN(CASE WHEN le.itemid = 51300 THEN le.valuenum END) AS wbc_min,
      MAX(CASE WHEN le.itemid = 51300 THEN le.valuenum END) AS wbc_max
    FROM ICUSTAYS c
    JOIN LABEVENTS le
      ON c.hadm_id = le.hadm_id
    WHERE c.icustay_id IN (", paste(block, collapse = ","), ")
      AND le.charttime BETWEEN c.intime AND DATE_ADD(c.intime, INTERVAL 24 HOUR)
      AND le.valuenum IS NOT NULL
    GROUP BY c.icustay_id;
  ")
  
  results[[i]] <- dbGetQuery(con, sql_block)
  
  current <- min(i + block_size - 1, total)
  if (current %% 1000 == 0 || current == total) {
    cat("Processed", current, "of", total, "ICU stays for labs\n")
    flush.console()
  }
}

labs_data <- bind_rows(results)
head(labs_data)


Total ICU stays to process for labs: 32000 
Processed 1000 of 32000 ICU stays for labs
Processed 2000 of 32000 ICU stays for labs
Processed 3000 of 32000 ICU stays for labs
Processed 4000 of 32000 ICU stays for labs
Processed 5000 of 32000 ICU stays for labs
Processed 6000 of 32000 ICU stays for labs
Processed 7000 of 32000 ICU stays for labs
Processed 8000 of 32000 ICU stays for labs
Processed 9000 of 32000 ICU stays for labs
Processed 10000 of 32000 ICU stays for labs
Processed 11000 of 32000 ICU stays for labs
Processed 12000 of 32000 ICU stays for labs
Processed 13000 of 32000 ICU stays for labs
Processed 14000 of 32000 ICU stays for labs
Processed 15000 of 32000 ICU stays for labs
Processed 16000 of 32000 ICU stays for labs
Processed 17000 of 32000 ICU stays for labs
Processed 18000 of 32000 ICU stays for labs
Processed 19000 of 32000 ICU stays for labs
Processed 20000 of 32000 ICU stays for labs
Processed 21000 of 32000 ICU stays for labs
Processed 22000 of 32000 ICU stays for la

,icustay_id,lactate_mean,lactate_min,lactate_max,creatinine_mean,creatinine_min,creatinine_max,wbc_mean,wbc_min,wbc_max
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,200066,1.1,1.1,1.1,3.000000,3.0,3.0,NA,NA,NA
2,201239,NA,NA,NA,0.550000,0.5,0.6,NA,NA,NA
3,202248,NA,NA,NA,0.800000,0.8,0.8,NA,NA,NA
4,203487,1.4,1.0,1.8,1.283333,0.9,1.6,NA,NA,NA
5,204407,NA,NA,NA,8.400000,8.4,8.4,NA,NA,NA
6,204798,NA,NA,NA,0.600000,0.6,0.6,NA,NA,NA


In [5]:

# Merge the cohort with vital signs and laboratory data
icu_data <- cohort_base %>%
  left_join(vitals_data, by = "icustay_id") %>%
  left_join(labs_data, by = "icustay_id")

colnames(icu_data)

head(icu_data)

summary(icu_data)


[1] "subject_id"           "hadm_id"              "icustay_id"          
 [4] "intime"               "gender"               "age"                 
 [7] "hospital_expire_flag" "hr_mean"              "hr_min"              
[10] "hr_max"               "map_mean"             "map_min"             
[13] "map_max"              "rr_mean"              "rr_min"              
[16] "rr_max"               "spo2_mean"            "spo2_min"            
[19] "spo2_max"             "temp_mean"            "temp_min"            
[22] "temp_max"             "lactate_mean"         "lactate_min"         
[25] "lactate_max"          "creatinine_mean"      "creatinine_min"      
[28] "creatinine_max"       "wbc_mean"             "wbc_min"             
[31] "wbc_max"

,subject_id,hadm_id,icustay_id,intime,gender,age,hospital_expire_flag,hr_mean,hr_min,hr_max,⋯,temp_max,lactate_mean,lactate_min,lactate_max,creatinine_mean,creatinine_min,creatinine_max,wbc_mean,wbc_min,wbc_max
,<int>,<int>,<int>,<dttm>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,3,145834,211552,2101-10-20 19:10:11,M,76,0,111.78571,75,168,⋯,98.2,4.871429,2.1,8.8,2.466667,2.4,2.5,NA,NA,NA
2,4,185777,294638,2191-03-16 00:29:31,F,48,0,89.21739,74,111,⋯,99.4,NA,NA,NA,0.500000,0.5,0.5,NA,NA,NA
3,6,107064,228232,2175-05-30 21:30:54,F,66,0,84.16000,76,100,⋯,99.2,NA,NA,NA,10.000000,10.0,10.0,NA,NA,NA
4,9,150750,220597,2149-11-09 13:07:02,M,41,1,92.50000,82,111,⋯,100.2,2.333333,1.9,2.7,1.400000,1.4,1.4,NA,NA,NA
5,11,194540,229441,2178-04-16 06:19:32,F,50,0,84.95833,70,101,⋯,99.3,NA,NA,NA,0.700000,0.7,0.7,NA,NA,NA
6,12,112213,232669,2104-08-08 02:08:17,M,72,1,85.82857,71,105,⋯,99.8,10.140000,2.0,15.1,1.500000,1.3,1.7,NA,NA,NA


   subject_id       hadm_id         icustay_id    
 Min.   :    3   Min.   :100001   Min.   :200003  
 1st Qu.:13636   1st Qu.:124786   1st Qu.:225332  
 Median :27272   Median :150060   Median :250463  
 Mean   :36894   Mean   :149997   Mean   :250262  
 3rd Qu.:60256   3rd Qu.:175277   3rd Qu.:275204  
 Max.   :97321   Max.   :199999   Max.   :299999  
                                                  
     intime                          gender               age        
 Min.   :2100-06-09 01:39:43.00   Length:32000       Min.   : 18.00  
 1st Qu.:2125-08-31 15:46:43.25   Class :character   1st Qu.: 53.00  
 Median :2150-11-26 16:29:06.50   Mode  :character   Median : 66.00  
 Mean   :2150-11-09 05:38:29.93                      Mean   : 74.53  
 3rd Qu.:2176-03-18 13:40:02.25                      3rd Qu.: 78.00  
 Max.   :2205-10-24 21:53:20.00                      Max.   :307.00  
                                                                     
 hospital_expire_flag    hr_mean

In [6]:
saveRDS(icu_data, "icu_data.rds")


* subject_id: Unique patient identifier

* hadm_id: Hospital admission identifier

* icustay_id: ICU stay identifier

* intime: ICU admission time

* gender: Patient biological sex

* age: Age at ICU admission

* Heart rate (bpm):

* hr_mean: Average heart rate

* hr_min: Lowest heart rate

* hr_max: Highest heart rate

* Mean arterial pressure (mmHg):

* map_mean: Average arterial pressure

* map_min: Lowest arterial pressure

* map_max: Highest arterial pressure

* Respiratory rate (breaths/min):

* rr_mean: Average respiratory rate

* rr_min: Lowest respiratory rate

* rr_max: Highest respiratory rate

* Oxygen saturation (%):

* spo2_mean: Average oxygen saturation

* spo2_min: Lowest oxygen saturation

* spo2_max: Highest oxygen saturation

* Temperature (°C or °F):

* temp_mean: Average body temperature

* temp_min: Lowest body temperature

* temp_max: Highest body temperature

* Lactate (mmol/L):

* lactate_mean: Average lactate level

* lactate_min: Lowest lactate level

* lactate_max: Highest lactate level

* Creatinine (mg/dL):

* creatinine_mean: Average creatinine level

* creatinine_min: Lowest creatinine level

* creatinine_max: Highest creatatinine level

* White blood cells (×10⁹/L):

* wbc_mean: Average white blood cell count

* wbc_min: Lowest white blood cell count

* wbc_max: Highest white blood cell count